Transformer Encoder-Decoder
====


  - encoder self-attention: bidirectional, no causal mask
  - decoder self-attention: causal mask
  - decoder cross-attention: queries from decoder, keys/values from encoder
  - feed-forward network after attention
  - stacked encoder and decoder blocks

https://github.com/huggingface/transformers/blob/main/src/transformers/models/bart/modeling_bart.py

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
from tokenizers import Tokenizer

In [2]:
embedding_dim = 512
context_length = 1024
num_heads = 8
dropout_rate = 0.1

sqrt_d_model = embedding_dim ** 0.5

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_dir = repo_root / "data" / "tokenizers"
tokenizer_dir.mkdir(parents=True, exist_ok=True)

tokenizer_path = tokenizer_dir / "tatoeba_bpe_en_de.json"
print(f"Tokenizer path: {tokenizer_path}")

Tokenizer path: /home/pyml/git_kmehr/llm-recipes/data/tokenizers/tatoeba_bpe_en_de.json


In [3]:
# Load the tokenizer from disk, like we would in later notebooks.
tokenizer = Tokenizer.from_file(str(tokenizer_path))

n_vocab = tokenizer.get_vocab_size()
print(f"Vocabulary size: {n_vocab:,}")

pad_token_id = tokenizer.token_to_id("[PAD]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")
unk_token_id = tokenizer.token_to_id("[UNK]")
print(f"PAD token id: {pad_token_id}")
print(f"BOS token id: {bos_token_id}")
print(f"EOS token id: {eos_token_id}")
print(f"UNK token id: {unk_token_id}")


# running an example
text_en = "The encoder uses attention."
text_de = "Der Encoder verwendet Attention."

encoded_en = tokenizer.encode(text_en)
encoded_de = tokenizer.encode(text_de)

print("English tokens:", encoded_en.tokens)
print("English ids:", encoded_en.ids)
print("German tokens:", encoded_de.tokens)
print("German ids:", encoded_de.ids)

Vocabulary size: 16,000
PAD token id: 0
BOS token id: 1
EOS token id: 2
UNK token id: 3
English tokens: ['[BOS]', 'The', 'en', 'co', 'der', 'uses', 'attention', '.', '[EOS]']
English ids: [1, 220, 158, 331, 197, 5893, 4537, 16, 2]
German tokens: ['[BOS]', 'Der', 'En', 'co', 'der', 'verwendet', 'Atten', 'tion', '.', '[EOS]']
German ids: [1, 328, 375, 331, 197, 7735, 14963, 2596, 16, 2]


### 1. Prepare the input

In [4]:
input_texts = [
    "Transformers use attention to process the input sequence.",
    "The encoder uses attention to create a representation of the input sequence.",
]

encoded_batch = tokenizer.encode_batch(input_texts)
input_ids = [encoded.ids for encoded in encoded_batch]
print(f"input_ids lengths: {[len(ids) for ids in input_ids]}")

pad_token_id = tokenizer.token_to_id("[PAD]")

max_len = max(len(ids) for ids in input_ids)

padded_input_ids = [
    ids + [pad_token_id] * (max_len - len(ids))
    for ids in input_ids
]

input_ids_tensor = torch.tensor(padded_input_ids, dtype=torch.long)

lengths = torch.tensor([len(ids) for ids in input_ids])
positions = torch.arange(max_len).unsqueeze(0)

attention_mask = positions >= lengths.unsqueeze(1)

print("input_ids_tensor:", input_ids_tensor.shape)   # [batch_size, max_len]
print("attention_mask:", attention_mask.shape)       # [batch_size, max_len]

print("input_ids_tensor:\n", input_ids_tensor)
print("attention_mask:\n", attention_mask)


input_ids lengths: [15, 20]
input_ids_tensor: torch.Size([2, 20])
attention_mask: torch.Size([2, 20])
input_ids_tensor:
 tensor([[    1, 14393, 12809,    80,   554,  4537,   181, 10476,   180,   160,
           959, 11841,  1785,    16,     2,     0,     0,     0,     0,     0],
        [    1,   220,   158,   331,   197,  5893,  4537,   181, 14888,    62,
         12763,   400,   201,   180,   160,   959, 11841,  1785,    16,     2]])
attention_mask:
 tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False]])


In [5]:
tok_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, context_length, embedding_dim):
        super().__init__()
        position = torch.arange(context_length).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-torch.log(torch.tensor(10000.0)) / embedding_dim)
        )

        pe = torch.zeros(context_length, embedding_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)

    def forward(self, positions):
        return self.pe[positions]


pos_embeddings = SinusoidalPositionEmbedding(
    context_length=context_length, embedding_dim=embedding_dim
)

tok_emb = tok_embeddings(input_ids_tensor)

batch_size, seq_len = input_ids_tensor.shape

positions = torch.arange(seq_len, dtype=torch.long, device=input_ids_tensor.device)
pos_emb = pos_embeddings(positions)  # [seq_len, embedding_dim]
print(f"pos_emb.shape: {pos_emb.shape}")  # [seq_len, 768]

# scale and add positional embeddings to token embeddings
batch_x = tok_emb * sqrt_d_model + pos_emb
# The original Transformer does this because token embeddings and sinusoidal position embeddings are added directly.

print(f"Input x: {batch_x.shape}")

# note: we should also apply embedding dropout here, but we will skip it for simplicity.
# embedding_dropout = nn.Dropout(p=dropout_rate)
# batch_x = embedding_dropout(tok_emb * sqrt_d_model + pos_emb)

pos_emb.shape: torch.Size([20, 512])
Input x: torch.Size([2, 20, 512])


### 2. Multi-head Attention (MHA)

In [6]:
# combining three types of MHA into one class: standard, cross-attention, and causal attention
class MultiheadAttention(nn.Module):
    def __init__(
            self,
            embedding_dim,
            num_heads,
            context_length,
            is_crossattention=False,
            is_causal=False,
            dropout_rate=0.1
        ):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        assert not (
            is_crossattention and is_causal
        ), "Cross-attention should not use a causal mask"
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim
        self.head_dim = embedding_dim // num_heads
        self.is_crossattention = is_crossattention
        self.is_causal = is_causal

        # if self.is_crossattention:
        #     self.w_attn = nn.Linear(embedding_dim, 2 * embedding_dim)  # for k & v
        #     self.w_q = nn.Linear(embedding_dim, embedding_dim)
        # else:
        #     self.w_attn = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)

        self.attn_dropout = nn.Dropout(dropout_rate)

        if self.is_causal:
            mask = torch.triu(
                torch.ones(context_length, context_length, dtype=torch.bool),
                diagonal=1,
            )
            self.register_buffer("causal_mask", mask)

    def forward(self, x, kv_states=None, attention_mask=None):
        batch_size, target_len, _ = x.size()

        if self.is_crossattention:
            assert kv_states is not None, "kv_states must be provided for cross-attention"
        else:
            kv_states = x
        src_len = kv_states.size(1)

        # Linear projections
        # if self.is_crossattention:
        #     q = self.w_q(x)
        #     k, v = self.w_attn(kv_states).split(self.embedding_dim, dim=-1)
        # else:
        #     q, k, v = self.w_attn(x).split(self.embedding_dim, dim=-1)
        q = self.w_q(x)
        k = self.w_k(kv_states)
        v = self.w_v(kv_states)
        # current shapes: [batch_size, seq_len, embedding_dim]

        # split the embedding dimension into multiple heads
        # -> shape: [batch_size, num_heads, seq_len, head_dim]
        q = q.view(batch_size, target_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, src_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Causal mask
        if self.is_causal:
            assert target_len <= self.causal_mask.size(0), "sequence length exceeds context_length"
            assert src_len <= self.causal_mask.size(0), "source length exceeds context_length"
            causal_mask = self.causal_mask[:target_len, :src_len]
            attn_scores = attn_scores.masked_fill(
                causal_mask.unsqueeze(0).unsqueeze(0),
                float('-inf')
            )

        # attention mask (for padding tokens)
        if attention_mask is not None:
            assert attention_mask.shape == (batch_size, src_len)
            assert attention_mask.dtype == torch.bool

            # attention_mask shape: [batch_size, src_len]
            attn_scores = attn_scores.masked_fill(
                attention_mask.unsqueeze(1).unsqueeze(2),
                float("-inf"),
            )

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        output = torch.matmul(attn_weights, v)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, target_len, -1)
        output = self.w_o(output)

        return output
    
# testing:
mha_layer = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    dropout_rate=0.1
)
output_mha = mha_layer(batch_x)
print(f"Input shape: {batch_x.shape}")
print(f"MHA Output shape: {output_mha.shape}")

mha_layer_cross = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=True,
    is_causal=False,
    dropout_rate=0.1
)
output_mha_cross = mha_layer_cross(batch_x, kv_states=batch_x)
print(f"MHA (cross-attention) output shape: {output_mha_cross.shape}")

mha_layer_causal = MultiheadAttention(
    embedding_dim=embedding_dim,
    num_heads=8,
    context_length=context_length,
    is_crossattention=False,
    is_causal=True,
    dropout_rate=0.1
)

output_mha_causal = mha_layer_causal(batch_x, attention_mask=attention_mask)
print(f"MHA (causal) output shape: {output_mha_causal.shape}")

Input shape: torch.Size([2, 20, 512])
MHA Output shape: torch.Size([2, 20, 512])
MHA (cross-attention) output shape: torch.Size([2, 20, 512])
MHA (causal) output shape: torch.Size([2, 20, 512])


### 3. Transformer Encoder

In [7]:
# MHA -> dropout -> residual -> layer norm -> FFN -> dropout -> residual -> layer norm

class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, context_length, dropout_rate=0.1):
        super().__init__()
        self.mha = MultiheadAttention(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            context_length=context_length,
            is_crossattention=False,
            is_causal=False,
            dropout_rate=dropout_rate
        )
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),  # original transformer uses ReLU
            nn.Linear(4 * embedding_dim, embedding_dim),
        )
        self.resid_dropout = nn.Dropout(dropout_rate)
        self.ffn_dropout = nn.Dropout(dropout_rate)
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x, attention_mask=None):
        # Multi-head attention
        attn_output = self.mha(x, attention_mask=attention_mask)
        x = self.layer_norm1(x + self.resid_dropout(attn_output))  # residual connection + layer norm
        # Feed-forward network
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + self.ffn_dropout(ffn_output))  # residual connection + layer norm
        return x
    

# testing:
encoder_block = EncoderBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    context_length=context_length,
    dropout_rate=0.1
)


output_encoder_block = encoder_block(batch_x, attention_mask=attention_mask)
print(f"Encoder block output shape: {output_encoder_block.shape}")

Encoder block output shape: torch.Size([2, 20, 512])


## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py
 * HuggingFace GPT-2 PyTorch Implementation: https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py
 * HuggingFace GPT-2 config: https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/configuration_gpt2.py